In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import(
    accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report,roc_curve,auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from wordcloud import WordCloud
nltk.download('stopwords')
import os

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
import kagglehub
path = kagglehub.dataset_download("benhamner/nips-papers")

print("Path to dataset files:", path)

100%|██████████| 141M/141M [00:01<00:00, 110MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/benhamner/nips-papers/versions/2


In [ ]:
import os
print(os.listdir(path))

['authors.csv', 'papers.csv', 'database.sqlite', 'paper_authors.csv']


In [ ]:
df=pd.read_csv('/root/.cache/kagglehub/datasets/benhamner/nips-papers/versions/2/papers.csv')
df

,id,year,title,event_type,pdf_name,abstract,paper_text
0,1,1987,Self-Organization of Associative Database and ...,NaN,1-self-organization-of-associative-database-an...,Abstract Missing,767\n\nSELF-ORGANIZATION OF ASSOCIATIVE DATABA...
1,10,1987,A Mean Field Theory of Layer IV of Visual Cort...,NaN,10-a-mean-field-theory-of-layer-iv-of-visual-c...,Abstract Missing,683\n\nA MEAN FIELD THEORY OF LAYER IV OF VISU...
2,100,1988,Storing Covariance by the Associative Long-Ter...,NaN,100-storing-covariance-by-the-associative-long...,Abstract Missing,394\n\nSTORING COVARIANCE BY THE ASSOCIATIVE\n...
3,1000,1994,Bayesian Query Construction for Neural Network...,NaN,1000-bayesian-query-construction-for-neural-ne...,Abstract Missing,Bayesian Query Construction for Neural\nNetwor...
4,1001,1994,"Neural Network Ensembles, Cross Validation, an...",NaN,1001-neural-network-ensembles-cross-validation...,Abstract Missing,"Neural Network Ensembles, Cross\nValidation, a..."
...,...,...,...,...,...,...,...
7236,994,1994,Single Transistor Learning Synapses,NaN,994-single-transistor-learning-synapses.pdf,Abstract Missing,Single Transistor Learning Synapses\n\nPaul Ha...
7237,996,1994,"Bias, Variance and the Combination of Least Sq...",NaN,996-bias-variance-and-the-combination-of-least...,Abstract Missing,"Bias, Variance and the Combination of\nLeast S..."
7238,997,1994,A Real Time Clustering CMOS Neural Engine,NaN,997-a-real-time-clustering-cmos-neural-engine.pdf,Abstract Missing,A Real Time Clustering CMOS\nNeural Engine\nT....
7239,998,1994,Learning direction in global motion: two class...,NaN,998-learning-direction-in-global-motion-two-cl...,Abstract Missing,Learning direction in global motion: two\nclas...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7241 entries, 0 to 7240
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          7241 non-null   int64 
 1   year        7241 non-null   int64 
 2   title       7241 non-null   object
 3   event_type  2422 non-null   object
 4   pdf_name    7241 non-null   object
 5   abstract    7241 non-null   object
 6   paper_text  7241 non-null   object
dtypes: int64(2), object(5)
memory usage: 396.1+ KB


In [ ]:
df.describe()

,id,year
count,7241.000000,7241.000000
mean,3655.912167,2006.439718
std,2098.435219,8.759919
min,1.000000,1987.000000
25%,1849.000000,2000.000000
50%,3659.000000,2009.000000
75%,5473.000000,2014.000000
max,7284.000000,2017.000000


In [ ]:
df['event_type'].duplicated()

,event_type
0,False
1,True
2,True
3,True
4,True
...,...
7236,True
7237,True
7238,True
7239,True


In [ ]:
df['abstract'].value_counts()

,count
abstract,
Abstract Missing,3317
"We study the power of different types of adaptive (nonoblivious) adversaries in the setting of prediction with expert advice, under both full-information and bandit feedback. We measure the player's performance using a new notion of regret, also known as policy regret, which better captures the adversary's adaptiveness to the player's behavior. In a setting where losses are allowed to drift, we characterize ---in a nearly complete manner--- the power of adaptive adversaries with bounded memories and switching costs. In particular, we show that with switching costs, the attainable rate with bandit feedback is $T^{2/3}$. Interestingly, this rate is significantly worse than the $\sqrt{T}$ rate attainable with switching costs in the full-information case. Via a novel reduction from experts to bandits, we also show that a bounded memory adversary can force $T^{2/3}$ regret even in the full information case, proving that switching costs are easier to control than bounded memory adversaries. Our lower bounds rely on a new stochastic adversary strategy that generates loss processes with strong dependencies.",2
"An active learner is given a class of models, a large set of unlabeled examples, and the ability to interactively query labels of a subset of these examples; the goal of the learner is to learn a model in the class that fits the data well. Previous theoretical work has rigorously characterized label complexity of active learning, but most of this work has focused on the PAC or the agnostic PAC model. In this paper, we shift our attention to a more general setting -- maximum likelihood estimation. Provided certain conditions hold on the model class, we provide a two-stage active learning algorithm for this problem. The conditions we require are fairly general, and cover the widely popular class of Generalized Linear Models, which in turn, include models for binary and multi-class classification, regression, and conditional random fields. We provide an upper bound on the label requirement of our algorithm, and a lower bound that matches it up to lower order terms. Our analysis shows that unlike binary classification in the realizable case, just a single extraround of interaction is sufficient to achieve near-optimal performance in maximum likelihood estimation. On the empirical side, the recent work in (Gu et al. 2012) and (Gu et al. 2014) (on active linear and logistic regression) shows the promise of this approach.",2
"We develop a fully discriminative learning approach for supervised Latent Dirichlet Allocation (LDA) model using Back Propagation (i.e., BP-sLDA), which maximizes the posterior probability of the prediction variable given the input document. Different from traditional variational learning or Gibbs sampling approaches, the proposed learning method applies (i) the mirror descent algorithm for maximum a posterior inference and (ii) back propagation over a deep architecture together with stochastic gradient/mirror descent for model parameter estimation, leading to scalable and end-to-end discriminative learning of the model. As a byproduct, we also apply this technique to develop a new learning method for the traditional unsupervised LDA model (i.e., BP-LDA). Experimental results on three real-world regression and classification tasks show that the proposed methods significantly outperform the previous supervised topic models, neural networks, and is on par with deep neural networks.",1
"Infinite Hidden Markov Models (iHMM's) are an attractive, nonparametric generalization of the classical Hidden Markov Model which can automatically infer the number of hidden states in the system. However, due to the infinite-dimensional nature of the transition dynamics, performing inference in the iHMM is difficult. In this paper, we present an infinite-state Particle Gibbs (PG) algorithm to resample state trajectories for the iHMM. The proposed algorithm uses an efficient proposal optimized 

In [ ]:
a=df['abstract']=='Abstract Missing'
a.value_counts()

,count
abstract,
False,3924
True,3317


In [ ]:
df['year'].value_counts()

,count
year,
2017,679
2016,569
2014,411
2015,403
2012,368
2013,360
2011,306
2010,292
2009,262


In [ ]:
df.isnull().sum()

,0
id,0
year,0
title,0
event_type,4819
pdf_name,0
abstract,0
paper_text,0


In [ ]:
df.drop(columns=['id'],axis=1,inplace=True)
df.drop(columns=['event_type'],axis=1,inplace=True)
df.drop(columns=['abstract'],axis=1,inplace=True)

In [ ]:
df

,year,title,pdf_name,paper_text
0,1987,Self-Organization of Associative Database and ...,1-self-organization-of-associative-database-an...,767\n\nSELF-ORGANIZATION OF ASSOCIATIVE DATABA...
1,1987,A Mean Field Theory of Layer IV of Visual Cort...,10-a-mean-field-theory-of-layer-iv-of-visual-c...,683\n\nA MEAN FIELD THEORY OF LAYER IV OF VISU...
2,1988,Storing Covariance by the Associative Long-Ter...,100-storing-covariance-by-the-associative-long...,394\n\nSTORING COVARIANCE BY THE ASSOCIATIVE\n...
3,1994,Bayesian Query Construction for Neural Network...,1000-bayesian-query-construction-for-neural-ne...,Bayesian Query Construction for Neural\nNetwor...
4,1994,"Neural Network Ensembles, Cross Validation, an...",1001-neural-network-ensembles-cross-validation...,"Neural Network Ensembles, Cross\nValidation, a..."
...,...,...,...,...
7236,1994,Single Transistor Learning Synapses,994-single-transistor-learning-synapses.pdf,Single Transistor Learning Synapses\n\nPaul Ha...
7237,1994,"Bias, Variance and the Combination of Least Sq...",996-bias-variance-and-the-combination-of-least...,"Bias, Variance and the Combination of\nLeast S..."
7238,1994,A Real Time Clustering CMOS Neural Engine,997-a-real-time-clustering-cmos-neural-engine.pdf,A Real Time Clustering CMOS\nNeural Engine\nT....
7239,1994,Learning direction in global motion: two class...,998-learning-direction-in-global-motion-two-cl...,Learning direction in global motion: two\nclas...


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
stop_words=set(stopwords.words('english'))
lem=WordNetLemmatizer()
def advanced_clean_text(text):
  text = str(text).lower()

  text = re.sub(r'http\S+|www\S+|https\S+','',text)
  text=re.sub(r'@\w+|#\w+','',text)
  text=re.sub(r'\d+','',text)
  text = re.sub(r'[^a-zA-Z\s]', '', text)
  text=re.sub(r'\s+',' ',text).strip()
  words=text.split()
  words=[
      lem.lemmatize(word)
      for word in words
      if word not in stop_words and len(word)>2
  ]
  return ' '.join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
df['paper_text']=df['paper_text'].apply(advanced_clean_text)

In [ ]:
df['pdf_name']=df['pdf_name'].apply(advanced_clean_text)

In [ ]:
# df['title']=df['title'].apply(advanced_clean_text)

In [ ]:
df.shape

(7241, 4)

In [ ]:
a=[]
for i, j in df['paper_text'].items():
    if len(str(j).split()) < 1000:
        a.append(i)
df.drop(a,inplace=True)

In [ ]:
df.shape

(7144, 4)

In [ ]:
df['new_paper_text']=df['pdf_name']+" "+df['paper_text']

In [ ]:
new_df=df.drop(columns=['year'],axis=1,inplace=True)
new_df=df.drop(columns=['pdf_name','paper_text'])


In [ ]:
new_df

,title,new_paper_text
0,Self-Organization of Associative Database and ...,selforganizationofassociativedatabaseanditsapp...
1,A Mean Field Theory of Layer IV of Visual Cort...,ameanfieldtheoryoflayerivofvisualcortexanditsa...
2,Storing Covariance by the Associative Long-Ter...,storingcovariancebytheassociativelongtermpoten...
3,Bayesian Query Construction for Neural Network...,bayesianqueryconstructionforneuralnetworkmodel...
4,"Neural Network Ensembles, Cross Validation, an...",neuralnetworkensemblescrossvalidationandactive...
...,...,...
7236,Single Transistor Learning Synapses,singletransistorlearningsynapsespdf single tra...
7237,"Bias, Variance and the Combination of Least Sq...",biasvarianceandthecombinationofleastsquaresest...
7238,A Real Time Clustering CMOS Neural Engine,arealtimeclusteringcmosneuralenginepdf real ti...
7239,Learning direction in global motion: two class...,learningdirectioninglobalmotiontwoclassesofpsy...


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vector = cv.fit_transform(new_df['new_paper_text']).toarray()

In [ ]:
# vectorr = cv.fit_transform(df['title']).toarray()

In [ ]:
a=new_df['title'].(2)
for i in a:
  print(i)

nan
nan


In [ ]:
vector.shape

(7144, 5000)

In [ ]:
# vectorr.shape

In [ ]:
vector

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 3, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
# vectorr

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vector)

In [ ]:
cleaned_titles = new_df['title'].apply(advanced_clean_text)
index_value = new_df[cleaned_titles == 'selforganization associative database application'].index[0]
index_value

np.int64(0)

In [ ]:
def recommend(df):
    index = new_df[cleaned_titles == df].index[0]
    distances = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1])
    for i in distances[1:6]:
        print(new_df.iloc[i[0]]['title'])

In [ ]:
recommend('selforganization associative database application')

Off-Road Obstacle Avoidance through End-to-End Learning
Investment Learning with Hierarchical PSOMs
Visual gesture-based robot guidance with a modular neural system
Semi-Supervised Learning in Gigantic Image Collections
Learning to Traverse Image Manifolds


In [ ]:
index = new_df[cleaned_titles == 'selforganization associative database application'].index[0]

distances = sorted(
    list(enumerate(similarity[index])),
    reverse=True,
    key=lambda x: x[1]
)

for i in distances[1:6]:
    print(new_df.iloc[i[0]]['title'], i[1])

Off-Road Obstacle Avoidance through End-to-End Learning 0.4067050364037248
Investment Learning with Hierarchical PSOMs 0.40367375922881343
Visual gesture-based robot guidance with a modular neural system 0.39553389262743477
Semi-Supervised Learning in Gigantic Image Collections 0.3914176788088222
Learning to Traverse Image Manifolds 0.3818529951116088


In [ ]:
 import pickle

In [ ]:
pickle.dump(new_df,open('paper.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))